# All-gene transcript concordance by biotype (publication figure)

Two-panel publication-ready figure:
- **Left** — Ensembl perspective: per Ensembl biotype, how well are Ensembl genes
  represented in CAT? RBH-paired genes carry their actual tx concordance; genes with
  no CAT pair in a given assembly count as *No match*.
- **Right** — CAT perspective: per CAT biotype, how well are CAT genes represented in
  Ensembl? RBH-paired genes carry their actual tx concordance (symmetric); CAT-only
  genes count as *No match*.

Bars are **percentage-normalised** per biotype row so the concordance profile can be
compared across biotypes with very different total counts. Total gene-assembly pairs
(*n*) are annotated on the right of each row.

**Data sources**
- `sankey_plus_divergence/links_per_assembly.tsv` — all RBH-paired genes (pass + fail)
  with columns `biotype` (Ensembl, grouped), `cat_biotype` (raw), `tx_concordance`.
- `gene_presence/sankey_level1_gene_presence.tsv` — cohort-level gene presence with
  `n_assemblies_ensembl_only` and `n_assemblies_cat_only` per gene.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# ── Font settings ─────────────────────────────────────────────────────────────
# TrueType fonts — text remains editable in Illustrator / Inkscape
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype']  = 42
matplotlib.rcParams['font.family']  = 'sans-serif'
matplotlib.rcParams['font.size']    = 8

# ── Paths ─────────────────────────────────────────────────────────────────────
RESULTS_DIR = Path('/hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results')
LINKS_DIR   = RESULTS_DIR / 'intermediate_spreadsheets' / 'sankey_plus_divergence'
GP_DIR      = RESULTS_DIR / 'intermediate_spreadsheets' / 'gene_presence'
OUTPUT_DIR  = Path('figures')
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Colour scheme ─────────────────────────────────────────────────────────────
CONCORDANCE_COLORS = {
    'full':    '#1b4f72',
    'partial': '#85c1e9',
    'none':    '#e74c3c',
}
CONCORDANCE_ORDER  = ['full', 'partial', 'none']
CONCORDANCE_LABELS = {'full': 'Full match', 'partial': 'Partial match', 'none': 'No match'}

BIOTYPE_ORDER  = ['protein_coding', 'lncRNA', 'pseudogene', 'other_ncRNA', 'other']
BIOTYPE_LABELS = {
    'protein_coding': 'Protein-coding',
    'lncRNA':         'lncRNA',
    'pseudogene':     'Pseudogene',
    'other_ncRNA':    'Other ncRNA',
    'other':          'Other',
}

def group_biotype(b: str) -> str:
    b = str(b or '').lower()
    if 'protein_coding' in b:
        return 'protein_coding'
    if 'lncrna' in b or 'lnc_rna' in b:
        return 'lncRNA'
    if 'pseudogene' in b or 'pseudogenic' in b:
        return 'pseudogene'
    if any(x in b for x in ['snrna', 'snorna', 'mirna', 'trna', 'rrna',
                             'ncrna', 'antisense', 'tec', 'guide_rna',
                             'scrna', 'vault_rna', 'y_rna']):
        return 'other_ncRNA'
    return 'other'

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
links = pd.read_csv(LINKS_DIR / 'links_per_assembly.tsv', sep='\t')
gp    = pd.read_csv(GP_DIR    / 'sankey_level1_gene_presence.tsv', sep='\t')

print(f'links_per_assembly.tsv : {len(links):,} rows, {links["assembly_accession"].nunique()} assemblies')
print(f'  rbh_status counts    : {dict(links["rbh_status"].value_counts())}')
print(f'  columns              : {list(links.columns)}')
print()
print(f'sankey_level1_gene_presence.tsv : {len(gp):,} genes')
print(f'  columns                       : {list(gp.columns)}')

In [ ]:
# ── Build cross-tabs ──────────────────────────────────────────────────────────

rbh_pairs = links[links['rbh_status'].isin(['pass', 'fail'])].copy()

# ── Ensembl perspective ───────────────────────────────────────────────────────
# 1. RBH-paired genes: biotype already grouped in 'biotype' column
ct_ens = (
    rbh_pairs
    .groupby(['biotype', 'tx_concordance'])
    .size()
    .unstack(fill_value=0)
    .reindex(index=BIOTYPE_ORDER, columns=CONCORDANCE_ORDER, fill_value=0)
)

# 2. Ensembl-only genes: n_assemblies_ensembl_only → all go to 'none'
gp['ens_bio_grp'] = gp['ensembl_biotype'].map(group_biotype)
ens_only_extra = (
    gp[gp['n_assemblies_ensembl_only'] > 0]
    .groupby('ens_bio_grp')['n_assemblies_ensembl_only']
    .sum()
    .reindex(BIOTYPE_ORDER, fill_value=0)
)
for bio in BIOTYPE_ORDER:
    ct_ens.loc[bio, 'none'] += int(ens_only_extra.loc[bio])

print('Ensembl cross-tab (gene-assembly counts):')
print(ct_ens)

# ── CAT perspective ───────────────────────────────────────────────────────────
# 1. RBH-paired genes: group cat_biotype from links
rbh_pairs = rbh_pairs.copy()
rbh_pairs['cat_bio_grp'] = rbh_pairs['cat_biotype'].map(group_biotype)
ct_cat = (
    rbh_pairs
    .groupby(['cat_bio_grp', 'tx_concordance'])
    .size()
    .unstack(fill_value=0)
    .reindex(index=BIOTYPE_ORDER, columns=CONCORDANCE_ORDER, fill_value=0)
)

# 2. CAT-only genes: n_assemblies_cat_only → all go to 'none'
gp['cat_bio_grp'] = gp['cat_biotype'].map(group_biotype)
cat_only_extra = (
    gp[gp['n_assemblies_cat_only'] > 0]
    .groupby('cat_bio_grp')['n_assemblies_cat_only']
    .sum()
    .reindex(BIOTYPE_ORDER, fill_value=0)
)
for bio in BIOTYPE_ORDER:
    ct_cat.loc[bio, 'none'] += int(cat_only_extra.loc[bio])

print('\nCAT cross-tab (gene-assembly counts):')
print(ct_cat)

In [ ]:
# ── Sanity checks ─────────────────────────────────────────────────────────────
n_asm = links['assembly_accession'].nunique()
print(f'Assemblies: {n_asm}')
print(f'Ensembl total gene-assembly pairs: {ct_ens.values.sum():,}')
print(f'  of which Ensembl-only additions: {int(ens_only_extra.sum()):,}')
print(f'CAT total gene-assembly pairs    : {ct_cat.values.sum():,}')
print(f'  of which CAT-only additions    : {int(cat_only_extra.sum()):,}')

In [ ]:
# ── Publication figure ────────────────────────────────────────────────────────

def draw_panel(
    ax, ct, panel_label,
    show_yticklabels=True,
    label_threshold_pct=6,
):
    """
    Draw one percentage-normalised stacked horizontal bar chart.

    Each row is normalised to 100 % so biotypes with very different total
    gene counts remain visually comparable.  The absolute total *n* is
    printed to the right of each row.
    """
    totals  = ct.sum(axis=1).values.astype(float)
    y_pos   = np.arange(len(BIOTYPE_ORDER))
    left    = np.zeros(len(BIOTYPE_ORDER))

    for conc in CONCORDANCE_ORDER:
        vals_abs = ct[conc].values.astype(float)
        vals_pct = np.where(totals > 0, vals_abs / totals * 100, 0.0)

        ax.barh(
            y_pos, vals_pct, left=left,
            color=CONCORDANCE_COLORS[conc],
            edgecolor='white', linewidth=0.4,
        )

        # Percentage labels inside segments wide enough to hold text
        for j, (pct, l, n_abs) in enumerate(zip(vals_pct, left, vals_abs)):
            if totals[j] == 0:
                continue
            if pct >= label_threshold_pct:
                ax.text(
                    l + pct / 2, j,
                    f'{pct:.0f}%',
                    ha='center', va='center',
                    fontsize=7, fontweight='bold', color='white',
                )
        left += vals_pct

    # Absolute-count annotation to the right of each full bar
    for j, (tot, y) in enumerate(zip(totals, y_pos)):
        ax.text(
            101, y, f'n\u2009=\u2009{int(tot):,}',
            va='center', ha='left',
            fontsize=7, color='#333333',
        )

    # Axes cosmetics
    ax.set_xlim(0, 100)
    ax.set_xlabel('Gene-assembly pairs (%)', fontsize=8)
    ax.set_xticks([0, 25, 50, 75, 100])
    ax.xaxis.set_tick_params(labelsize=7)

    ax.set_yticks(y_pos)
    if show_yticklabels:
        ax.set_yticklabels([BIOTYPE_LABELS[b] for b in BIOTYPE_ORDER], fontsize=8)
    else:
        ax.set_yticklabels([])

    ax.invert_yaxis()   # protein-coding at top
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Panel label (A / B) in upper-left corner
    ax.text(
        -0.12, 1.04, panel_label,
        transform=ax.transAxes,
        fontsize=11, fontweight='bold', va='top', ha='left',
    )


# ── Layout: 2 panels, shared y-axis ──────────────────────────────────────────
# Width: ~90 mm per panel + space for n= annotations → 7.5 inches total
# Height: 5 biotypes × ~0.55 inch row-height → 3.5 inches
fig, (ax_ens, ax_cat) = plt.subplots(
    1, 2,
    figsize=(8.5, 3.4),
    sharey=True,
    gridspec_kw={'wspace': 0.35},
)

draw_panel(ax_ens, ct_ens, 'A', show_yticklabels=True)
draw_panel(ax_cat, ct_cat, 'B', show_yticklabels=False)

ax_ens.set_title('Ensembl genes vs CAT', fontsize=9, fontweight='bold', pad=6)
ax_cat.set_title('CAT genes vs Ensembl', fontsize=9, fontweight='bold', pad=6)

# ── Shared legend below panels ────────────────────────────────────────────────
legend_handles = [
    mpatches.Patch(facecolor=CONCORDANCE_COLORS[c], label=CONCORDANCE_LABELS[c], edgecolor='white')
    for c in CONCORDANCE_ORDER
]
fig.legend(
    handles=legend_handles,
    loc='lower center',
    ncol=3,
    fontsize=8,
    frameon=False,
    bbox_to_anchor=(0.46, -0.07),
)

plt.savefig(OUTPUT_DIR / 'figure_all_genes_biotype_concordance.pdf', bbox_inches='tight')
plt.savefig(OUTPUT_DIR / 'figure_all_genes_biotype_concordance.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved to {OUTPUT_DIR}/figure_all_genes_biotype_concordance.(pdf|png)')

In [ ]:
# ── Summary tables ────────────────────────────────────────────────────────────

def make_summary(ct, perspective_label):
    totals = ct.sum(axis=1)
    rows = []
    for bio in BIOTYPE_ORDER:
        tot = int(totals[bio])
        row = {'perspective': perspective_label, 'biotype': BIOTYPE_LABELS[bio], 'n_total': tot}
        for conc in CONCORDANCE_ORDER:
            n = int(ct.loc[bio, conc])
            row[f'n_{conc}'] = n
            row[f'pct_{conc}'] = round(n / tot * 100, 1) if tot > 0 else 0.0
        rows.append(row)
    return pd.DataFrame(rows)

summary = pd.concat([
    make_summary(ct_ens, 'Ensembl'),
    make_summary(ct_cat, 'CAT'),
], ignore_index=True)

display(summary)
summary.to_csv(OUTPUT_DIR / 'figure_all_genes_biotype_concordance_summary.tsv', sep='\t', index=False)
print('Summary saved.')